In [3]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Sample Grasps
Sample antipodal grasps on the object mesh.

In [2]:
import sys

sys.path.append("..")

import meshio
from config import ROBOTIQ_HANDE_GRIPPER
from sampler import GraspSampler

from meshgraphnet.utils import msh_to_trimesh

msh = meshio.read("../meshes/test/msh/Bushing3_cg1.msh")
mesh = msh_to_trimesh(msh)
gripper = ROBOTIQ_HANDE_GRIPPER()
sampler = GraspSampler(mesh=mesh, gripper=gripper, mu=0.1)
grasps = sampler.sample(n_samples=10, debug=False)



Sampled 7 antipodal grasps, checking for collisions...
3 valid grasps found after collision checking.


# Find Optimal Grasps
Load GNN model and select the best grasp (with highest score) according to the model's predictions.

In [16]:
import sys

sys.path.append("..")

import meshio
import torch
from config import ROBOTIQ_HANDE_GRIPPER
from optimizer import GNNBasedGraspOptimizer

from meshgraphnet.nets import EncodeProcessDecode
from meshgraphnet.normalizer import Normalizer

gripper = ROBOTIQ_HANDE_GRIPPER()

msh = meshio.read("../meshes/test/msh/L-Bracket4_cg1.msh")

checkpoint = torch.load(
    "../models/Model0.pth",
    map_location=torch.device("cpu"),
    weights_only=True,
)
model_state_dict = checkpoint["model_state_dict"]
params = checkpoint["params"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
normalizer = Normalizer(
    num_features=params["node_dim"],
    num_categorical=params["num_categorical"],
    device=device,
    stats=checkpoint["stats"],
)
model = EncodeProcessDecode(
    node_dim=params["node_dim"],
    edge_dim=params["edge_dim"],
    output_dim=params["output_dim"],
    latent_dim=params["latent_dim"],
    message_passing_steps=params["message_passing_steps"],
    use_layer_norm=params["use_layer_norm"],
).to(device)
model.load_state_dict(model_state_dict)
model.eval()

optimizer = GNNBasedGraspOptimizer(gripper, model, normalizer, device=device)
grasps = optimizer.optimize(msh, mu=0.1, k=20)



Sampled 286 antipodal grasps, checking for collisions...
151 valid grasps found after collision checking.


# Visualize Grasps
Visualize the optimal grasps on the object mesh.

In [ ]:
from sampler import GraspSampler

from meshgraphnet.utils import msh_to_trimesh

mesh = msh_to_trimesh(msh)

score, grasp = grasps[5]
print("Best grasp score:", score)
print("Best grasp:", grasp)

sampler = GraspSampler(mesh=mesh, gripper=gripper, mu=0.1)
scene = sampler.visualize_grasp(grasp)

Best grasp score: 35691.44140625
Best grasp: Grasp(
  pose=Pose(pos=[ 1.30104261e-18 -5.19842852e-02  1.55770730e-01], quat=[ 0.96592583  0.          0.         -0.25881905]),
  width=0.015,
  c1=Contact(pos=[-0.0075      0.02101571  0.02933102], normal=[-1.  0.  0.], mu=0.1),
  c2=Contact(pos=[0.0075     0.02101571 0.02933102], normal=[ 1. -0. -0.], mu=0.1),
  wrench=[ 0.75 , -0.601,  0.278, -0.   , -0.   ,  0.   ]
)


In [29]:
import trimesh
from scipy.spatial.transform import Rotation as R

base_pose = grasp.pose
print("Base:")
print("   Position:", base_pose.pos)
print("   Orientation (quaternion):", base_pose.quat)

fingertip_pose = base_pose.se3() @ gripper.tf_base_to_fingertip()
print("Fingertip:")
print("   Position:", fingertip_pose[:3, 3])
print("   Orientation (quaternion):", R.from_matrix(fingertip_pose[:3, :3]).as_quat())

x_dir = fingertip_pose[:3, 0]

# Find the contact points on the object mesh
intersector = trimesh.ray.ray_triangle.RayMeshIntersector(mesh)
locs1, index_ray1, index_tri1 = intersector.intersects_location(
    [fingertip_pose[:3, 3]], [x_dir], multiple_hits=True
)
locs2, index_ray2, index_tri2 = intersector.intersects_location(
    [fingertip_pose[:3, 3]], [-x_dir], multiple_hits=True
)

scene = trimesh.Scene()
scene.add_geometry(mesh, node_name="object")

base_visual = trimesh.primitives.Sphere(radius=0.005, center=base_pose.pos)
base_visual.visual.face_colors = [132, 32, 55, 255]
scene.add_geometry(base_visual, node_name="base")

fingertip_visual = trimesh.primitives.Sphere(radius=0.005, center=fingertip_pose[:3, 3])
fingertip_visual.visual.face_colors = [132, 32, 55, 255]
scene.add_geometry(fingertip_visual, node_name="fingertip")

contact_visual = trimesh.primitives.Sphere(radius=0.005, center=locs1[0])
contact_visual.visual.face_colors = [32, 132, 55, 255]
scene.add_geometry(contact_visual, node_name="contact")

contact_visual2 = trimesh.primitives.Sphere(radius=0.005, center=locs2[0])
contact_visual2.visual.face_colors = [32, 132, 55, 255]
scene.add_geometry(contact_visual2, node_name="contact2")

scene.show(viewer="gl")

Base:
   Position: [ 0.         -0.1372359   0.09762219]
   Orientation (quaternion): [0.        0.8660254 0.5       0.       ]
Fingertip:
   Position: [ 0.         -0.01079619  0.02462219]
   Orientation (quaternion): [0.        0.8660254 0.5       0.       ]


SceneViewer(width=1800, height=1350)

In [ ]:
from config import ROBOTIQ_HANDE_GRIPPER

gripper = ROBOTIQ_HANDE_GRIPPER()
gripper.show_box_fingers(0.025)